# Analiza streszczeń literatury młodzieżowej

### Import bibliotek

In [ ]:
import os
import json
import string
import morfeusz2
import numpy as np
import pandas as pd
import operator as op
import itertools as it
from wordcloud import WordCloud
from matplotlib import pyplot as plt
from nltk import ngrams
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

In [ ]:
import nltk
nltk.download('punkt_tab')
if "." not in nltk.data.path:
    nltk.data.path.append(".")

## Zdefiniowanie funkcji pomocniczych

In [ ]:
def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))

In [ ]:
def create_dir(path):
    if not os.path.exists(path):
        os.mkdir(path)

## Przygotwanie danych wejściowych

#### Utworzenie korpusu dokumentów

In [ ]:
corpus_dir = "./literature"
corpus = PlaintextCorpusReader(corpus_dir, r'.*\.txt')
files_names = corpus.fileids()
titles = [file_name.removesuffix('.txt') for file_name in files_names]

In [ ]:
documents_dict = {title: corpus.raw(file_name) for title, file_name in zip(titles, files_names)}
print_dict(documents_dict)

In [ ]:
documents_params = {}
for title, document in documents_dict.items():
    documents_params[title] = {
        "pre": len(word_tokenize(document, language="polish"))
    }
print_dict(documents_params)

### Utworzenie macierzy częstości (przed wstępnym przetwarzaniem)

In [ ]:
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df_pre.columns = ["content"]
display(documents_df_pre)

In [ ]:
cv = CountVectorizer()
tv = TfidfTransformer()

In [ ]:
matrix_tf = cv.fit_transform(documents_df_pre["content"])
sparsity_tf = 1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))

In [ ]:
matricies_params = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparsity": sparsity_tf,
    }
}
print_dict(matricies_params)

### Wstępne przygotowanie dokumentów

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords_file.close()
stopwords = stopwords[4:]

In [ ]:
def lemmatize(text):
    lemmatizer = morfeusz2.Morfeusz()
    segments = it.groupby(lemmatizer.analyse(text), op.itemgetter(0,1))
    def disambiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _,) in group)
        _, lemma = min(pairs)
        return lemma.split(":")[0]
    return " ".join(filter(str.isalpha, (disambiguate(group) for _, group in segments)))

In [ ]:
for title in documents_dict:
    documents_dict[title] = documents_dict[title].lower()
    documents_dict[title] = "".join(filter(lambda char: char not in string.punctuation, documents_dict[title]))
    documents_dict[title] = lemmatize(documents_dict[title])
    documents_dict[title] = " ".join(filter(lambda word: word not in stopwords, word_tokenize(documents_dict[title], language="polish")))
print_dict(documents_dict)

In [ ]:
for title, document in documents_dict.items():
    documents_params[title]["post"] = len(word_tokenize(document, language="polish"))
print_dict(documents_params)

In [ ]:
documents_params_df = pd.DataFrame.from_dict(documents_params, orient="index")
documents_params_df.columns = ["pre", "post"]
documents_params_df["change"] = documents_params_df["post"] - documents_params_df["pre"]
documents_params_df["change_percent"] = documents_params_df["change"] / documents_params_df["pre"] * 100
display(documents_params_df)


### Utworzenie macierzy częstości (po wstępnym przetwarzaniu)

In [ ]:
documents_df_post = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df_post.columns = ["content"]
display(documents_df_post)

In [ ]:
matrix_tf = cv.fit_transform(documents_df_post["content"])
#matrix_tfidf = tv.fit_transform(documents_df_post["content"])
sparsity_tf = 1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))

In [ ]:
matricies_params["post"] = {
    "words": matrix_tf.shape[1],
    "sparsity": sparsity_tf,
}
print_dict(matricies_params)

In [22]:
matricies_params_df = pd.DataFrame.from_dict(matricies_params, orient="index")
display(matricies_params_df)

,words,sparsity
pre,7469,0.909560
post,4675,0.893123


### Przygotowanie katalogów naw wyniki

In [ ]:
create_dir("./wordclouds")
create_dir("./wordclouds/pre")
create_dir("./wordclouds/post")
create_dir("./topic_modeling")
for t in [10, 6, 4, 3]:
    create_dir(f"./topic_modeling/{t}")
    create_dir(f"./topic_modeling/{t}/topics")
    create_dir(f"./topic_modeling/{t}/documents")
create_dir("./clustering")
create_dir("./ngrams")
for n in [1, 2, 3]:
    create_dir(f"./ngrams/{n}")

## Chmury tagów

In [24]:
wordcloud = WordCloud(
    background_color="white",
    max_words=5000,
    colormap="PuBuGn_r"
)

In [28]:
for index, row in documents_df_pre.iterrows():
    wordcloud.generate(row["content"])
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.title(index)
    plt.axis("off")
    plt.savefig(f"./wordclouds/pre/{index}.png", bbox_inches="tight")
    plt.close()

In [27]:
for index, row in documents_df_post.iterrows():
    wordcloud.generate(row["content"])
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.title(index)
    plt.axis("off")
    plt.savefig(f"./wordclouds/post/{index}.png", bbox_inches="tight")
    plt.close()

## Analiza tematyk

In [29]:
def plot_topics(model, feature_names, n_top_words, title, subplots, colors):
    n = len(model.components_)
    fig, axes = plt.subplots(*subplots, figsize=(30, 15), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7, color = colors[topic_idx % len(colors)])
        ax.set_title(f"Temat {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.savefig(f"./topic_modeling/{n}/topics/{title}.png", bbox_inches="tight")
    plt.close()

In [ ]:
def plot_documents(model, matrix, title, colors):
    n = len(model.components_)
    documents_topics = pd.DataFrame(model.transform(matrix), columns=[f"Temat {i+1}" for i in range(n)], index=documents_df_post.index)
    plt.figure(figsiz=(7,4))
    documents_topics.plot(kind="barh", stacked=True, color=colors, legend=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"./topic_modeling/{n}/documents/{title}.csv")
    plt.close()

In [ ]:
n_topics = [10, 6, 4, 3]
feature_names = cv.get_feature_names_out()
n_top_words = 20
subplots = [(2,5), (2,3), (1,4), (1,3)]
colors = ["seagreen", "royalblue", "violet", "mediumturqoise", "slateblue", "tomato", "lawngreen", "wheat", "indigo", "indianred"]

In [35]:
def analyze_lda(n_topics, matrix, feature_names, n_top_words, subplots, colors):
    lda = LatentDirichletAllocation(
        n_components= n_topics,
        random_state= 0,
        max_iter= 5,
        learning_method= 'online',
        learning_offset= 50.0,
    )
    lda.fit(matrix)
    plot_topics(lda, feature_names, n_top_words, f"LDA - {n} tematów", subplots, colors)
    plot_documents(lda, matrix, f"LDA = {n} tematów", colors)


In [36]:
def analyze_nmf_fn(n_topics, matrix, feature_names, n_top_words, subplots, colors):
    nmf = NMF(
        n_components= n_topics,
        random_state= 1,
        alpha_H= .00005,
        alpha_W= .0005,
        l1_ratio= .5,
    )
    nmf.fit(matrix)
    plot_topics(nmf, feature_names, n_top_words, f"NMF (FN) - {n_topics} tematów", subplots, colors)
    plot_documents(nmf, matrix, f"NMF (FN) = {n_topics} tematów", colors)

In [37]:
def analyze_nmf_kl(n_topics, matrix, feature_names, n_top_words, subplots, colors):
    nmf = NMF(
        n_components= n_topics,
        random_state= 1,
        beta_loss= 'kullback-leibler',
        solver= 'mu',
        max_iter= 1000,
        alpha_H= .00005,
        alpha_W= .0005,
        l1_ratio= .5,
    )
    nmf.fit(matrix)
    plot_topics(nmf, feature_names, n_top_words, f"NMF (KL) - {n_topics} tematów", subplots, colors)
    plot_documents(nmf, matrix, f"NMF (KL) = {n_topics} tematów", colors)

In [ ]:
for n, s in zip(n_topics, subplots):
    analyze_lda(n, matrix_tf, feature_names, n_top_words, s, colors)
    analyze_nmf_fn(n, matrix_tf, feature_names, n_top_words, s, colors)
    analyze_nmf_kl(n, matrix_tf, feature_names, n_top_words, s, colors)